# 精确f696三资产H5未来赢家与原现金提案的区间几何

只读封存Qlib DatasetH标签、f696五输出与819个原账户窗口。未来赢家仅作诊断标签；不产生交易规则、拟合或账户。

In [1]:
import os
os.environ["MLFLOW_ALLOW_FILE_STORE"]="true"
import hashlib,json
from pathlib import Path
import numpy as np,pandas as pd,qlib
from qlib.data import D
from qlib.workflow import R

root=Path.cwd();card=root/"research/stage1_f696_h5_asset_winner_geometry_20261024"
plan_path=card/"plan.json";plan=json.loads(plan_path.read_text());src=plan["sources"]
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan["frozen"] and all(plan["budget"][k]==0 for k in ("new_market_GETs","new_model_fits","new_native_accounts","new_bootstraps","new_Recorders"))
a=root/src["labels_card"];f=root/src["f696_card"];b=root/src["account_card"]
for file,key in [(a/"summary.json","labels_summary_sha256"),(f/"summary.json","f696_summary_sha256"),(b/"summary.json","account_summary_sha256"),(root/src["regime_plan"],"regime_plan_sha256")]:assert sha(file)==src[key],key
sa=json.loads((a/"summary.json").read_text());sf=json.loads((f/"summary.json").read_text());sb=json.loads((b/"summary.json").read_text())
assert sa["nas_evidence"]["sha256"]==src["labels_NAS_sha256"] and sa["nas_evidence"]["size_bytes"]==src["labels_NAS_size_bytes"] and sa["nas_readback"]["verified"]
assert sf["nas_evidence"]["sha256"]==src["f696_NAS_sha256"] and sf["nas_evidence"]["size_bytes"]==src["f696_NAS_size_bytes"]
assert sb["nas_readback"]["verified"] and sb["nas_readback"]["sha256"]==src["account_NAS_sha256"]
aa=a/"qlib_recorder"/src["labels_experiment_id"]/src["labels_recorder_id"]/"artifacts/inputs/dataset"
fa=f/"qlib_recorder"/src["f696_experiment_id"]/src["f696_recorder_id"]/"artifacts/inputs/frame"
ba=b/"qlib_recorder"/src["account_experiment_id"]/src["account_recorder_id"]/"artifacts/analysis/rolling126"
assert sha(aa)==src["labels_dataset_sha256"] and sha(fa)==src["f696_frame_sha256"] and sha(ba)==src["account_rolling_sha256"]
assert sha(root/src["qlib_provider"]/"qlib_store_manifest.json")==src["provider_manifest_sha256"]
qlib.init(provider_uri=str((root/src["qlib_provider"]).resolve()),region="us",expression_cache=None,dataset_cache=None,kernels=1)
with R.uri_context(str((a/"qlib_recorder").resolve())):
 rec=R.get_recorder(experiment_id=src["labels_experiment_id"],recorder_id=src["labels_recorder_id"])
 dataset=rec.load_object("inputs/dataset")
with R.uri_context(str((f/"qlib_recorder").resolve())):
 rec=R.get_recorder(experiment_id=src["f696_experiment_id"],recorder_id=src["f696_recorder_id"])
 frame=rec.load_object("inputs/frame").sort_values("trade_date").reset_index(drop=True)
with R.uri_context(str((b/"qlib_recorder").resolve())):
 rec=R.get_recorder(experiment_id=src["account_experiment_id"],recorder_id=src["account_recorder_id"])
 rolling=rec.load_object("analysis/rolling126")
train=dataset.prepare("train",col_set=["label","meta"]);held=dataset.prepare("held",col_set=["label","meta"])
train=train.loc[train[("meta","train_mature")].astype(bool)];held=held.loc[held[("meta","held_mature")].astype(bool)]
assert len(train)==3014 and len(held)==940 and len(frame)==944 and len(rolling)==819
held_dates=pd.DatetimeIndex(held.index.get_level_values("datetime"));fdates=pd.DatetimeIndex(frame.signal_date)
assert held_dates.is_unique and fdates.is_unique and held_dates.equals(fdates[:940])
assets=["CASH","SOXX","SOXL"]
def classify(cash,relative):
 values=np.column_stack([cash,np.zeros(len(cash)),relative])
 maxval=np.max(values,axis=1)
 tie=np.isclose(values,maxval[:,None],rtol=0,atol=1e-12).sum(axis=1)>1
 winner=np.array(assets,dtype=object)[np.argmax(values,axis=1)]
 winner[tie]="TIE"
 return winner
train_cash=train[("label","cash_advantage_H5")].to_numpy(float);train_rel=train[("label","SOXL_relative_H5")].to_numpy(float)
held_cash=held[("label","cash_advantage_H5")].to_numpy(float);held_rel=held[("label","SOXL_relative_H5")].to_numpy(float)
assert np.isfinite(train_cash).all() and np.isfinite(train_rel).all() and np.isfinite(held_cash).all() and np.isfinite(held_rel).all()
train_winner=classify(train_cash,train_rel);held_winner=classify(held_cash,held_rel)
assert int(np.sum((train_cash<0)&(train_rel<0)))==79
assert int(np.sum((held_cash<0)&(held_rel<0)))==33
assert int(np.sum(train_winner=="SOXX"))==79 and int(np.sum(held_winner=="SOXX"))==33
winner=np.r_[held_winner,["UNKNOWN"]*4]
frame["H5_winner_hindsight"]=winner
frame["cash_advantage_H5"]=np.r_[held_cash,[np.nan]*4]
frame["SOXL_relative_H5"]=np.r_[held_rel,[np.nan]*4]
assert int(frame.H5_winner_hindsight.eq("UNKNOWN").sum())==4
calendar=pd.DatetimeIndex(D.calendar(start_time="2011-01-03",end_time="2026-10-09",freq="day"))
close=D.features(["SOXX"],["$close"],start_time="2011-01-03",end_time="2026-10-09",freq="day")["$close"].droplevel("instrument").sort_index()
assert close.index.equals(calendar) and fdates.isin(calendar).all()
def cohort(g):
 known=g.loc[~g.H5_winner_hindsight.eq("UNKNOWN")]
 out={"sessions":len(g),"mature_winners":len(known),"unknown":len(g)-len(known),
      "winner_counts":{x:int(known.H5_winner_hindsight.eq(x).sum()) for x in [*assets,"TIE"]},
      "SOXX_beats_cash_rows":int(known.cash_advantage_H5.lt(0).sum()),
      "SOXX_beats_SOXL_rows":int(known.SOXL_relative_H5.lt(0).sum()),
      "mean_f696_cash_weight":float(g.cash_weight.mean())}
 out["mean_f696_cash_by_future_winner"]={x:(float(known.loc[known.H5_winner_hindsight.eq(x),"cash_weight"].mean()) if known.H5_winner_hindsight.eq(x).any() else None) for x in assets}
 return out
annual=[];quarter=[]
for year,g in frame.groupby(pd.to_datetime(frame.trade_date).dt.year,sort=True):annual.append({"year":int(year),"partial_endpoint":year in (2023,2026),**cohort(g)})
for (year,q),g in frame.groupby([pd.to_datetime(frame.trade_date).dt.year,pd.to_datetime(frame.trade_date).dt.quarter],sort=True):quarter.append({"year":int(year),"quarter":int(q),**cohort(g)})
assert len(annual)==4 and len(quarter)==16 and sum(x["mature_winners"] for x in annual)==940
windows=[]
for i in range(819):
 rw=rolling.iloc[i];g=frame.iloc[i:i+126]
 assert pd.Timestamp(g.trade_date.iloc[0])==pd.Timestamp(rw.first_trade)
 signal=pd.Timestamp(g.signal_date.iloc[0]);pos=calendar.get_loc(signal);assert pos>=20
 r5=float(close.iloc[pos]/close.iloc[pos-5]-1);r20=float(close.iloc[pos]/close.iloc[pos-20]-1)
 state="uptrend" if r5>=0 and r20>=0 else "rebound" if r5>=0 else "pullback" if r20>=0 else "decline"
 known=g.loc[~g.H5_winner_hindsight.eq("UNKNOWN")]
 row={"year":int(pd.Timestamp(rw.first_trade).year),"start_state":state,"outcome":"weak" if rw.relative_log_excess<0 else "good" if rw.relative_log_excess>0 else "tie",
      "mature_days":len(known),"mean_f696_cash":float(g.cash_weight.mean())}
 row.update({x:float(known.H5_winner_hindsight.eq(x).mean()) for x in assets})
 windows.append(row)
w=pd.DataFrame(windows)
assert w.outcome.value_counts().to_dict()=={"weak":564,"good":255}
assert w.mature_days.min()>=122 and w.mature_days.max()==126
cells=[]
for (state,year),g in w.groupby(["start_state","year"],sort=True):
 a=g[g.outcome.eq("weak")];b=g[g.outcome.eq("good")]
 row={"start_state":state,"year":int(year),"weak_windows":len(a),"good_windows":len(b),"matched":bool(len(a) and len(b))}
 if row["matched"]:row["weak_minus_good"]={name:float(a[name].mean()-b[name].mean()) for name in [*assets,"mean_f696_cash"]}
 cells.append(row)
matched=[x for x in cells if x["matched"]]
assert len(matched)==12 and all(x["good_windows"]==0 for x in cells if x["year"]==2023)
weights=np.array([min(x["weak_windows"],x["good_windows"]) for x in matched],float)
matched_diff={name:float(np.average([x["weak_minus_good"][name] for x in matched],weights=weights)) for name in [*assets,"mean_f696_cash"]}
result={"status":"read_only_f696_H5_three_asset_future_winner_geometry_no_policy",
        "plan_sha256":sha(plan_path),"label_recorder_id":src["labels_recorder_id"],"f696_recorder_id":src["f696_recorder_id"],"account_recorder_id":src["account_recorder_id"],
        "train_mature_rows":len(train),"train_future_SOXX_joint_winner":int(np.sum(train_winner=="SOXX")),
        "held_mature_rows":len(held),"held_future_SOXX_joint_winner":int(np.sum(held_winner=="SOXX")),
        "full_f696_sessions":len(frame),"rolling126_windows":len(rolling),"rolling_mature_min_max":[int(w.mature_days.min()),int(w.mature_days.max())],
        "annual":annual,"quarter":quarter,"matched_state_year_cells":len(matched),"matched_weak_minus_good":matched_diff,"cells":cells,
        "new_market_GETs":0,"new_model_fits":0,"new_native_accounts":0,"new_bootstraps":0,"new_Recorders":0,
        "tradable_winner_signal":False,"independent_validation":False,"SOXX_goal_achieved":False,
        "limits":["H5 winners are outcome-defined and overlapping, not observable at the f696 or b472 decision clock",
                  "Saved f696 weights are proposals before candidate-specific own-state risk, fees and share execution",
                  "A future winner mix cannot be used to select a threshold or prove a fee-net SOXX account"]}
(card/"result.json").write_text(json.dumps(result,ensure_ascii=False,indent=2)+"\n")
print({"annual":[{k:x[k] for k in ("year","mature_winners","winner_counts","mean_f696_cash_weight") } for x in annual],"matched":matched_diff})


{'annual': [{'year': 2023, 'mature_winners': 249, 'winner_counts': {'CASH': 110, 'SOXX': 9, 'SOXL': 130, 'TIE': 0}, 'mean_f696_cash_weight': 0.37302566706888474}, {'year': 2024, 'mature_winners': 252, 'winner_counts': {'CASH': 106, 'SOXX': 8, 'SOXL': 138, 'TIE': 0}, 'mean_f696_cash_weight': 0.49697808467771093}, {'year': 2025, 'mature_winners': 250, 'winner_counts': {'CASH': 91, 'SOXX': 8, 'SOXL': 151, 'TIE': 0}, 'mean_f696_cash_weight': 0.4161464366638335}, {'year': 2026, 'mature_winners': 189, 'winner_counts': {'CASH': 72, 'SOXX': 8, 'SOXL': 109, 'TIE': 0}, 'mean_f696_cash_weight': 0.3239695222308778}], 'matched': {'CASH': -0.0066834432444716485, 'SOXX': -0.00967241961116855, 'SOXL': 0.016355862855640176, 'mean_f696_cash': 0.027536160940432933}}


## 中文结论

只读封存的原39项双H5标签Qlib DatasetH、精确f696 A0五输出及原0bp Tiered账户819个滚动窗口。将相同信号日上现金、SOXX、SOXL相对SOXX的未来H5毛开盘价格边际记为 `cash_advantage_H5 / 0 / SOXL_relative_H5`，仅作**事后赢家标签**；3014个成熟训练日与940个成熟复用评价日可计算，最后4个f696评价日保持未知。全年、16季度及全部819个126日窗口均纳入，窗口最多4个未来赢家未知。

SOXX同时胜过两者只占训练 **79/3014**、评价 **33/940**。2023部分年249个成熟日的赢家：SOXL **130**、现金 **110**、SOXX **9**；2024为138/106/8，2025为151/91/8，2026部分年为109/72/8（顺序SOXL／现金／SOXX）。2023在SOXL未来最优日，f696事前原提案平均现金 **0.4016**，在现金未来最优日则为 **0.3427**。同事前SOXX趋势状态及首交易年内的12个可配对好坏窗口格子，坏减好SOXL未来赢家占比描述差+0.0164、原现金提案+0.0275；2023的四格仍全部无好窗口对照。

这些未来赢家大量重叠，不能在09:29或09:46直接知道，均值差也不是因子效果或账户反事实。它解释了上一个“SOXX同时胜出”联合回归为什么稀有且没有正分数：在需要减少现金机会成本的历史里，更常需要区分**SOXL胜现金**与**现金保护有效**，SOXX本身很少是三者中的最高H5毛收益。下一项可证伪接收任务应是成对的SOXL相对现金优势，但先要冻结模型、对照和时间门，再以完整f696自身0bp费用后账户作最终经济检验。

首次notebook错误地要求包括Qlib读取预算在内所有预算值为0，来源尚未读取即停止；`attempt1_preflight_failure.json`记录该实现错误，修复只改为检查经济消耗为0，未改赢家定义或区间。决定 `descriptive_pairwise_SOXL_vs_cash_receiver_priority_no_policy`，SOXX v2目标未达成。
